# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library for the [FAIR^2 Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset object
dataset = mlc.Dataset(croissant_url)
# Get metadata attributes
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', None)}\n\nDescription: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview

Review available record sets, fields, and their IDs (all references by `@id`).

In [ ]:
# Display all record sets with their @id, name, and contained field @id's

print("Available RecordSets:")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"     - Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
    print()

# Optionally, also print out the @id for columns for each field
# Record the first record set and a numeric field for later use
record_sets = dataset.metadata.record_sets

if len(record_sets) > 0:
    main_record_set_id = record_sets[0].id
    main_record_set_name = record_sets[0].name
    # Collect all numeric field IDs for sample analysis
    numeric_fields = [field for field in record_sets[0].fields if getattr(field, 'data_type', None) in ['Float', 'Integer', 'Number']]
    if numeric_fields:
        main_numeric_field_id = numeric_fields[0].id
    else:
        main_numeric_field_id = record_sets[0].fields[0].id
else:
    main_record_set_id = None
    main_record_set_name = None
    main_numeric_field_id = None

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames. Use the record set and field `@id`s from the overview.

In [ ]:
# Build a dictionary of DataFrames: key is RecordSet @id

import collections

dataframes = collections.OrderedDict()
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for rs_id in record_set_ids:
    # Pull all records for this RecordSet
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet {rs_id}")
    else:
        print(f"No records found for RecordSet {rs_id}")

# Show available DataFrames
print(f"Available RecordSet IDs in dataframes: {list(dataframes.keys())}")

# Print columns for the main record set
if main_record_set_id in dataframes:
    print(f"The columns for RecordSet {main_record_set_id} are:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("The main record set could not be loaded as a DataFrame.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distribution, and grouping by key attributes.

In [ ]:
# For demonstration, pick a numeric field in the main record set (by @id)

import numpy as np

if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    numeric_field = main_numeric_field_id
    print(f"Using RecordSet: {main_record_set_id}, Numeric field: {numeric_field}")
    
    # Remove records with missing numeric values
    df_analysis = df.copy()
    if numeric_field in df_analysis.columns:
        # Try to convert column to numeric
        df_analysis[numeric_field] = pd.to_numeric(df_analysis[numeric_field], errors='coerce')
        df_analysis = df_analysis.dropna(subset=[numeric_field])
        threshold = df_analysis[numeric_field].mean() # Or set any appropriate threshold
        filtered_df = df_analysis[df_analysis[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df[[numeric_field]].head())
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select another field for grouping (e.g. the first string/categorical field)
        group_field = None
        for field in df_analysis.columns:
            if field != numeric_field and df_analysis[field].dtype == 'object':
                group_field = field
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print(f"No suitable group field found in columns for grouping beyond numeric field {numeric_field}.")
    else:
        print(f"The numeric field {numeric_field} is not available in the DataFrame columns.")
else:
    print("No main DataFrame for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    numeric_field = main_numeric_field_id
    if numeric_field in df.columns:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        plt.figure(figsize=(8, 5))
        df[numeric_field].hist(bins=40)
        plt.title(f"Distribution of '{numeric_field}' in RecordSet '{main_record_set_id}'")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
    else:
        print(f"Numeric field '{numeric_field}' not found in main DataFrame.")
else:
    print("Main DataFrame for visualization not found.")

## 6. Conclusion

In this notebook, we have:
- Loaded FAIR^2 Croissant metadata and records using `mlcroissant`
- Displayed available record sets, field names, and types via their `@id`
- Extracted record sets into pandas DataFrames, exploring columns and basic records
- Performed exploratory filtering, normalization, and grouping using field `@id`
- Visualized the distribution of a numeric field

You can continue with further domain-specific analysis or export results as needed!